## 목표: GPT Fine-tuning 시 Validation Loss 측정 구현


### 1. 라이브러리 및 환경 설정

In [ ]:
import os
import sys
import math
import torch
import wandb
import logging
import datasets
import argparse
import evaluate
import transformers

from typing import Optional
from itertools import chain
from dataclasses import dataclass, field

from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator
)
from transformers.trainer_utils import get_last_checkpoint

wandb.init(project='Hanghae99-ai')
wandb.run.name = 'gpt-finetuning'


### 2. Argument 정의

In [ ]:
@dataclass
class Arguments:
    model_name_or_path: Optional[str] = field(default='gpt2')
    torch_dtype: Optional[str] = field(default='float32', metadata={'choices': ['auto', 'bfloat16', 'float16', 'float32']})
    
    dataset_name: Optional[str] = field(default='your_dataset')
    dataset_config_name: Optional[str] = field(default=None)
    block_size: int = field(default=1024)
    num_workers: Optional[int] = field(default=2)

parser = HfArgumentParser((Arguments, TrainingArguments))
args, training_args = parser.parse_args_into_dataclasses()


### 3. 로깅설정

In [ ]:
logger = logging.getLogger()

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)

if training_args.should_log:
    transformers.utils.logging.set_verbosity_info()

log_level = training_args.get_process_log_level()
logger.setLevel(log_level)
datasets.utils.logging.set_verbosity(log_level)
transformers.utils.logging.set_verbosity(log_level)

transformers.utils.logging.enable_default_handler()
transformers.utils.logging.enable_explicit_format()

logger.info(f"Training/evaluation parameters {training_args}")


### 4. 데이터 로드 및 분리

In [ ]:
raw_datasets = load_dataset(
    args.dataset_name,
    args.dataset_config_name
)

# ⚠️ Validation Split이 없는 경우 자동 분할
if "validation" not in raw_datasets:
    raw_datasets = raw_datasets["train"].train_test_split(test_size=0.1)


### 5. Tokenizer & Model 설정

In [ ]:
config = AutoConfig.from_pretrained(args.model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(
    args.model_name_or_path,
    config=config,
    torch_dtype=args.torch_dtype
)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}"

if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))


### 6. Tokenization & Grouping

In [ ]:
column_names = list(raw_datasets["train"].features)
text_column_name = "text" if "text" in column_names else column_names[0]

def tokenize_function(examples):
    return tokenizer(examples[text_column_name])

with training_args.main_process_first(desc="Tokenizing dataset"):
    tokenized_datasets = raw_datasets.map(
        tokenize_function,
        batched=True,
        num_proc=args.num_workers,
        remove_columns=column_names
    )

block_size = min(args.block_size, tokenizer.model_max_length)

def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = (len(concatenated[list(examples.keys())[0]]) // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

with training_args.main_process_first(desc="Grouping texts"):
    lm_datasets = tokenized_datasets.map(
        group_texts,
        batched=True,
        num_proc=args.num_workers
    )


### 7. Trainer 준비 ( Validation 포함)


In [ ]:
train_dataset = lm_datasets["train"]
eval_dataset = lm_datasets["test"]  # ✅ validation dataset 지정

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # ✅ validation dataset 추가
    tokenizer=tokenizer,
    data_collator=default_data_collator
)


### 8. Trainning

In [ ]:
checkpoint = None
last_checkpoint = get_last_checkpoint(training_args.output_dir)
if training_args.resume_from_checkpoint is not None:
    checkpoint = training_args.resume_from_checkpoint
else:
    checkpoint = last_checkpoint

train_result = trainer.train(resume_from_checkpoint=checkpoint)

trainer.save_model()
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()


### 9. 평가 및 로그 기록

In [ ]:
eval_metrics = trainer.evaluate()
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)


### 10. 결과 공유: Wandb

In [ ]:
# GPT Fine-tuning with Validation

## ✅ Wandb Log
[링크: wandb 로그 공유된 URL]

## ✅ 주요 지표
- Train Loss: XX.XX
- Eval Loss: XX.XX

## ✅ 실행 방법
```bash
python train.py \
  --model_name_or_path=gpt2 \
  --dataset_name=your_dataset \
  --output_dir=./gpt-finetune \
  --per_device_train_batch_size=2 \
  --per_device_eval_batch_size=2 \
  --evaluation_strategy=epoch \
  --num_train_epochs=3 \
  --logging_steps=100 \
  --save_strategy=epoch \
  --report_to=wandb
